In [1]:
import os

# Install the specified wheel file using pip
os.system("pip install rtdl_num_embeddings-0.0.11-py3-none-any.whl")

# Rename the file 'onebap.bin' to 'onebap.so'
os.system("mv onebap.bin onebap.so")

# Import necessary libraries
import pickle  # For serializing and deserializing Python objects
import torch  # PyTorch for deep learning
import torch.nn as nn  # Neural network modules
import torch.nn.functional as F  # Functions for neural networks
import torch.optim  # Optimizers for training
from torch.utils.data import Dataset, DataLoader, TensorDataset  # Data handling utilities
from sklearn.model_selection import train_test_split  # Splitting data into training and testing sets

from sklearn.metrics import r2_score  # For calculating R-squared score
import pandas as pd  # Data manipulation and analysis
import math  # Mathematical functions
import numpy as np  # Numerical computations
from tqdm import tqdm  # Progress bar for loops
import polars as pl  # Alternative to pandas for dataframes
from collections import OrderedDict  # Ordered dictionary for maintaining insertion order
import sys  # System-specific parameters and functions

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
sys.path.append(project_root)
from models.tabm_reference import Model, make_parameter_groups  # Custom model and utility functions

import warnings  # For suppressing warnings
warnings.filterwarnings("ignore")  # Ignore all warnings

import joblib  # For saving and loading Python objects

# Import LightningModule from PyTorch Lightning for easier training
from pytorch_lightning import LightningModule

import warnings

In [2]:

file_path = os.path.expanduser('../processed_data/Processed_data_1695.parquet')
# read the data
data = pl.read_parquet(file_path)

# Add a 'row_id' column, starting from 0 and sequentially incrementing
data = data.with_row_count("row_id", offset=0)

# Add lagged columns for each responder (responder_{idx}_lag_1)
for idx in range(9):
    lag_col_name = f"responder_{idx}_lag_1"  # Name of the lagged column
    original_col_name = f"responder_{idx}"  # Name of the original column
    # Create the lagged column by shifting the original column by 1 row
    data = data.with_columns(pl.col(original_col_name).shift(1).alias(lag_col_name))

# Print the data after adding 'row_id' and lagged columns
print("\nData after adding 'row_id' and lagged columns:")
data


Data after adding 'row_id' and lagged columns:


row_id,date_id,time_id,symbol_id,weight,feature_00,feature_01,feature_02,feature_03,feature_04,feature_05,feature_06,feature_07,feature_08,feature_09,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,…,feature_60,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,responder_0,responder_1,responder_2,responder_3,responder_4,responder_5,responder_6,responder_7,responder_8,responder_0_lag_1,responder_1_lag_1,responder_2_lag_1,responder_3_lag_1,responder_4_lag_1,responder_5_lag_1,responder_6_lag_1,responder_7_lag_1,responder_8_lag_1
u32,i16,i16,i8,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
0,1695,0,0,3.373552,2.776059,1.035769,1.790385,1.915069,1.765592,-0.105391,-0.16963,-0.345125,0.03763,11.0,7.0,76.0,-1.029333,0.271748,-0.640703,-0.688929,-0.763758,-0.664928,-1.701001,-1.246213,1.171051,-0.151066,1.780195,0.592178,2.011824,1.034978,1.035668,1.118305,0.616451,-0.932135,-1.078001,-0.154166,…,-0.019777,1.419774,-0.401114,-0.292407,-0.196831,-1.569348,-2.423477,-0.725855,0.203034,-0.451438,-0.841593,0.31118,-0.575253,0.171726,0.140167,1.225793,0.929635,0.067034,0.06966,-0.106951,-0.133057,0.037213,0.094861,-1.123129,-0.055226,0.167316,-1.281933,-0.084505,null,null,null,null,null,null,null,null,null
1,1695,0,1,2.802384,1.901147,1.029203,2.146977,2.811388,1.423654,-0.105936,-0.173366,-0.382062,0.045153,11.0,7.0,76.0,-1.263074,0.253009,-0.599086,-0.688929,-0.633995,-0.664928,-1.906501,-0.91079,0.886705,0.012082,1.116906,0.85608,1.12455,0.723127,-1.920028,-0.053758,1.010441,-0.559132,-0.940507,0.010087,…,-0.079671,1.419774,-0.49611,-0.439725,-0.442559,-1.185965,-1.933517,-1.124566,0.131424,-0.604832,-0.760508,0.142326,-0.745543,0.171726,0.140167,-0.226083,-0.213198,-0.206452,-0.373929,-0.443276,-0.326311,-0.234482,0.262298,-1.037623,0.010625,0.730008,-1.589878,0.265547,-0.106951,-0.133057,0.037213,0.094861,-1.123129,-0.055226,0.167316,-1.281933,-0.084505
2,1695,0,2,2.506616,2.801315,1.376992,2.313733,2.135918,1.765473,-0.173767,-0.261385,-0.473618,0.018804,81.0,2.0,59.0,-0.99006,0.873093,-0.564999,-0.688929,-0.389982,-0.664928,-1.932154,-0.971507,0.023837,-0.145004,0.877403,0.370831,0.23665,-0.146222,0.580706,0.660125,0.125094,-0.627553,-0.784622,-0.183313,…,1.418136,1.419774,-0.363737,-0.314952,-0.355761,-1.836319,-1.781683,-0.99552,-0.030046,-0.476795,-0.888185,1.50716,-0.264173,0.171726,0.140167,1.517169,1.503251,0.094979,0.109544,0.351735,0.075186,0.683174,-1.139817,1.508688,-1.052497,-1.758799,2.506598,-2.090755,-0.443276,-0.326311,-0.234482,0.262298,-1.037623,0.010625,0.730008,-1.589878,0.265547
3,1695,0,3,1.868808,2.492323,1.293729,2.051276,2.601039,2.115296,-0.137015,-0.15458,-0.39843,0.058264,4.0,3.0,11.0,-1.047425,1.116742,-0.303803,-0.688929,-0.510801,-0.664928,-1.360154,-1.818448,-0.282608,-0.068771,0.591393,0.9088,0.051669,-0.261807,-0.01116,-0.169174,-0.219266,-0.642355,-0.530943,-0.037706,…,-0.0473,1.419774,-0.315971,-0.243883,-0.276656,-2.015491,-2.266636,-1.011813,0.711239,-0.346282,-1.06627,0.20662,-0.518951,0.171726,0.140167,0.425632,0.424689,-0.129259,-0.146268,0.047888,-0.01461,-0.334644,-0.0426,-0.569183,-0.244618,-0.055909,-0.570931,-0.120065,0.351735,0.075186,0.683174,-1.139817,1.508688,-1.052497,-1.758799,2.506598,-2.090755
4,1695,0,4,2.826087,2.754668,0.981692,2.405406,2.504181,2.060546,-0.067486,-0.153702,-0.205151,0.021181,15.0,1.0,9.0,-0.840723,1.097859,-0.419687,-0.688929,-0.834465,-0.664928,-1.323989,-1.13767,-1.158483,0.721674,0.600382,0.

In [3]:
file_path = os.path.expanduser('../models/model0108.pkl')

# Load the saved model and data statistics
with open(file_path, "rb") as fp:
    # Unpack the data from the file into multiple variables
    [data_stats, xgb_model, xgb_model2, features, lgb1, cat1, xgb1, _, _] = pickle.load(fp)

# Generate a list of features, excluding the 61st feature
feature_list = [f"feature_{idx:02d}" for idx in range(79) if idx != 61]

# Target column name
target_col = "responder_6"

# Test feature list, including original features and lag features
feature_test = feature_list + [f"responder_{idx}_lag_1" for idx in range(9)]

# Categorical feature columns
feature_cat = ["feature_09", "feature_10", "feature_11"]

# Continuous feature columns, excluding categorical features from the test features
feature_cont = [item for item in feature_test if item not in feature_cat]

# Batch size
batch_size = 8192

# Standardized feature columns, including original features (excluding categorical features) and lag features
std_feature = [i for i in feature_list if i not in feature_cat] + [f"responder_{idx}_lag_1" for idx in range(9)]

# Extract mean and standard deviation from the data statistics
means = data_stats['mean']
stds = data_stats['std']

In [4]:


# Standardize continuous features using mean and standard deviation
def standardize(df, feature_cols, means, stds):
    return df.with_columns([
        ((pl.col(col) - means[col]) / stds[col]).alias(col) for col in feature_cols
    ])

# Mapping for encoding categorical features
category_mappings = {
    'feature_09': {2: 0, 4: 1, 9: 2, 11: 3, 12: 4, 14: 5, 15: 6, 25: 7, 26: 8, 30: 9, 34: 10, 42: 11, 44: 12, 46: 13, 49: 14, 50: 15, 57: 16, 64: 17, 68: 18, 70: 19, 81: 20, 82: 21},
    'feature_10': {1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 10: 7, 12: 8},
    'feature_11': {9: 0, 11: 1, 13: 2, 16: 3, 24: 4, 25: 5, 34: 6, 40: 7, 48: 8, 50: 9, 59: 10, 62: 11, 63: 12, 66: 13, 76: 14, 150: 15, 158: 16, 159: 17, 171: 18, 195: 19, 214: 20, 230: 21, 261: 22, 297: 23, 336: 24, 376: 25, 388: 26, 410: 27, 522: 28, 534: 29, 539: 30},
    'symbol_id': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18, 19: 19, 20: 20, 21: 21, 22: 22, 23: 23, 24: 24, 25: 25, 26: 26, 27: 27, 28: 28, 29: 29, 30: 30, 31: 31, 32: 32, 33: 33, 34: 34, 35: 35, 36: 36, 37: 37, 38: 38},
    'time_id': {i: i for i in range(968)}
}

# Encode categorical columns using a mapping
def encode_column(df, column, mapping):
    max_value = max(mapping.values())  # Get the maximum value in the mapping

    def encode_category(category):
        return mapping.get(category, max_value + 1)  # Map category to encoded value, default to max_value + 1 if not found
    
    return df.with_columns(
        pl.col(column).map_elements(encode_category).alias(column)  # Apply encoding to the column
    )

# Custom R² loss function
class R2Loss(nn.Module):
    def __init__(self):
        super(R2Loss, self).__init__()

    def forward(self, y_pred, y_true):
        mse_loss = torch.sum((y_pred - y_true) ** 2)  # Mean squared error
        var_y = torch.sum(y_true ** 2)  # Variance of true values
        loss = mse_loss / (var_y + 1e-38)  # R² loss
        return loss

# Neural network model for tabular data
class NN(LightningModule):
    def __init__(self, n_cont_features, cat_cardinalities, n_classes, lr, weight_decay):
        super().__init__()
        self.save_hyperparameters()  # Save hyperparameters for logging
        self.k = 16  # Number of outputs per input
        self.model = Model(  # Define the model architecture
            n_num_features=n_cont_features,
            cat_cardinalities=cat_cardinalities,
            n_classes=n_classes,
            backbone={
                'type': 'MLP',
                'n_blocks': 3,
                'd_block': 512,
                'dropout': 0.25,
            },
            bins=None,
            num_embeddings=None,
            arch_type='tabm',
            k=self.k,
        )
        self.lr = lr  # Learning rate
        self.weight_decay = weight_decay  # Weight decay for regularization
        self.training_step_outputs = []  # Store training outputs
        self.validation_step_outputs = []  # Store validation outputs
        self.loss_fn = R2Loss()  # Use R² loss

    def forward(self, x_cont, x_cat):
        return self.model(x_cont, x_cat).squeeze(-1)  # Forward pass

    def training_step(self, batch):
        x_cont, x_cat, y, w, w_y = batch
        x_cont = x_cont + torch.randn_like(x_cont) * 0.02  # Add noise for regularization
        y_hat = self(x_cont, x_cat)  # Model predictions
        loss = self.loss_fn(y_hat.flatten(0, 1), y.repeat_interleave(self.k))  # Compute loss
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True, batch_size=x_cont.size(0))  # Log loss
        self.training_step_outputs.append((y_hat.mean(1), y, w))  # Store outputs
        return loss

    def validation_step(self, batch):
        x_cont, x_cat, y, w, w_y = batch
        x_cont = x_cont + torch.randn_like(x_cont) * 0.02  # Add noise for regularization
        y_hat = self(x_cont, x_cat)  # Model predictions
        loss = self.loss_fn(y_hat.flatten(0, 1), y.repeat_interleave(self.k))  # Compute loss
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True, logger=True, batch_size=x_cont.size(0))  # Log loss
        self.validation_step_outputs.append((y_hat.mean(1), y, w))  # Store outputs
        return loss

    def on_validation_epoch_end(self):
        """Calculate validation R² at the end of the epoch."""
        y = torch.cat([x[1] for x in self.validation_step_outputs]).cpu().numpy()  # True values
        prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()  # Predictions
        weights = torch.cat([x[2] for x in self.validation_step_outputs]).cpu().numpy()  # Weights
        val_r_square = r2_val(y, prob, weights)  # Compute R²
        self.log("val_r_square", val_r_square, prog_bar=True, on_step=False, on_epoch=True)  # Log R²
        self.validation_step_outputs.clear()  # Clear outputs

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(make_parameter_groups(self.model), lr=self.lr, weight_decay=self.weight_decay)  # Define optimizer
        return {'optimizer': optimizer}

    def on_train_epoch_end(self):
        """Calculate training R² at the end of the epoch."""
        y = torch.cat([x[1] for x in self.training_step_outputs]).cpu().numpy()  # True values
        prob = torch.cat([x[0] for x in self.training_step_outputs]).detach().cpu().numpy()  # Predictions
        weights = torch.cat([x[2] for x in self.training_step_outputs]).cpu().numpy()  # Weights
        train_r_square = r2_val(y, prob, weights)  # Compute R²
        self.log("train_r_square", train_r_square, prog_bar=True, on_step=False, on_epoch=True)  # Log R²
        self.training_step_outputs.clear()  # Clear outputs

        # Print metrics
        epoch = self.trainer.current_epoch
        metrics = {k: v.item() if isinstance(v, torch.Tensor) else v for k, v in self.trainer.logged_metrics.items()}
        formatted_metrics = {k: f"{v:.5f}" for k, v in metrics.items()}
        print(f"Epoch {epoch}: {formatted_metrics}")

# Custom arguments for the model
class custom_args():
    def __init__(self):
        self.usegpu = True  # Use GPU if available
        self.gpuid = 0  # GPU ID
        self.seed = 42  # Random seed
        self.model = 'nn'  # Model type
        self.use_wandb = False  # Disable Weights & Biases logging
        self.project = 'js-tabm-with-lags'  # Project name
        self.dname = "./input_df/"  # Data directory
        self.loader_workers = 10  # Number of data loader workers
        self.bs = 8192  # Batch size
        self.lr = 1e-3  # Learning rate
        self.weight_decay = 8e-4  # Weight decay
        self.n_cont_features = 84  # Number of continuous features
        self.n_cat_features = 5  # Number of categorical features
        self.n_classes = None  # Number of classes (not used)
        self.cat_cardinalities = [23, 10, 32, 40, 969]  # Cardinalities of categorical features
        self.patience = 7  # Early stopping patience
        self.max_epochs = 10  # Maximum number of epochs
        self.N_fold = 5  # Number of folds for cross-validation

# Initialize custom arguments
my_args = custom_args()

# Set device (GPU if available, else CPU)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

file_path = os.path.expanduser('../models/tabepoch9.ckpt')
# Load trained model from checkpoint
modeltrans = NN.load_from_checkpoint(file_path).to(device)

# Global variables for lag features
lags_: pl.DataFrame | None = None
lags_history = None


In [5]:

# Ignore all warnings to avoid cluttering the output
warnings.filterwarnings('ignore')

# Set Pandas to display all columns when printing DataFrames
pd.options.display.max_columns = None

# Configuration class for model settings
class CONFIG2:
    seed = 42  # Random seed for reproducibility
    target_col = "responder_6"  # Target column for prediction
    # Feature columns to be used in the model
    feature_cols = [f"feature_{idx:02d}" for idx in range(79)] + [f"responder_{idx}_lag_1" for idx in range(9)]
    
    # Paths to pre-trained models or model checkpoints
    model_paths = [
        "nnmodels",  # Path to neural network models
        "result.pkl",  # Path to a pre-trained model saved as a pickle file
    ]

# Additional feature columns for XGBoost models
xgb_feature_cols = ["symbol_id", "time_id"] + CONFIG2.feature_cols

# Custom R2 metric for validation
def r2_val(y_true, y_pred, sample_weight):
    """
    Calculate the weighted R-squared (R2) metric.
    Args:
        y_true: Ground truth values.
        y_pred: Predicted values.
        sample_weight: Weights for each sample.
    Returns:
        Weighted R-squared value.
    """
    r2 = 1 - np.average((y_pred - y_true) ** 2, weights=sample_weight) / (np.average((y_true) ** 2, weights=sample_weight) + 1e-38)
    return r2

# Neural Network model class
class NN(LightningModule):
    def __init__(self, input_dim, hidden_dims, dropouts, lr, weight_decay):
        """
        Initialize the neural network model.
        Args:
            input_dim: Input dimension (number of features).
            hidden_dims: List of hidden layer dimensions.
            dropouts: List of dropout rates for each hidden layer.
            lr: Learning rate for the optimizer.
            weight_decay: Weight decay (L2 regularization) for the optimizer.
        """
        super().__init__()
        self.save_hyperparameters()  # Save hyperparameters for logging
        layers = []  # List to store layers of the network
        in_dim = input_dim  # Input dimension for the first layer
        for i, hidden_dim in enumerate(hidden_dims):
            layers.append(nn.BatchNorm1d(in_dim))  # Batch normalization
            if i > 0:
                layers.append(nn.SiLU())  # SiLU activation function
            if i < len(dropouts):
                layers.append(nn.Dropout(dropouts[i]))  # Dropout layer
            layers.append(nn.Linear(in_dim, hidden_dim))  # Linear layer
            in_dim = hidden_dim  # Update input dimension for the next layer
        layers.append(nn.Linear(in_dim, 1))  # Output layer
        layers.append(nn.Tanh())  # Tanh activation for the output
        self.model = nn.Sequential(*layers)  # Combine all layers into a sequential model
        self.lr = lr  # Learning rate
        self.weight_decay = weight_decay  # Weight decay
        self.validation_step_outputs = []  # Store validation outputs for metric calculation

    def forward(self, x):
        """
        Forward pass of the model.
        Args:
            x: Input tensor.
        Returns:
            Scaled output tensor.
        """
        return 5 * self.model(x).squeeze(-1)  # Scale output and remove extra dimension

    def training_step(self, batch):
        """
        Training step for the model.
        Args:
            batch: Batch of data containing inputs, targets, and weights.
        Returns:
            Loss value.
        """
        x, y, w = batch  # Unpack batch
        y_hat = self(x)  # Forward pass
        loss = F.mse_loss(y_hat, y, reduction='none') * w  # Weighted MSE loss
        loss = loss.mean()  # Average loss
        self.log('train_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))  # Log training loss
        return loss

    def validation_step(self, batch):
        """
        Validation step for the model.
        Args:
            batch: Batch of data containing inputs, targets, and weights.
        Returns:
            Loss value.
        """
        x, y, w = batch  # Unpack batch
        y_hat = self(x)  # Forward pass
        loss = F.mse_loss(y_hat, y, reduction='none') * w  # Weighted MSE loss
        loss = loss.mean()  # Average loss
        self.log('val_loss', loss, on_step=False, on_epoch=True, batch_size=x.size(0))  # Log validation loss
        self.validation_step_outputs.append((y_hat, y, w))  # Store outputs for metric calculation
        return loss

    def on_validation_epoch_end(self):
        """
        Calculate validation metrics at the end of the epoch.
        """
        y = torch.cat([x[1] for x in self.validation_step_outputs]).cpu().numpy()  # Concatenate ground truth values
        if self.trainer.sanity_checking:  # Skip during sanity check
            prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()
        else:
            prob = torch.cat([x[0] for x in self.validation_step_outputs]).cpu().numpy()  # Concatenate predictions
            weights = torch.cat([x[2] for x in self.validation_step_outputs]).cpu().numpy()  # Concatenate weights
            val_r_square = r2_val(y, prob, weights)  # Calculate weighted R-squared
            self.log("val_r_square", val_r_square, prog_bar=True, on_step=False, on_epoch=True)  # Log R-squared
        self.validation_step_outputs.clear()  # Clear stored outputs

    def configure_optimizers(self):
        """
        Configure the optimizer and learning rate scheduler.
        Returns:
            Dictionary containing optimizer and scheduler.
        """
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)  # Adam optimizer
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)  # LR scheduler
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss',  # Monitor validation loss for scheduling
            }
        }

    def on_train_epoch_end(self):
        """
        Log metrics at the end of each training epoch.
        """
        if self.trainer.sanity_checking:  # Skip during sanity check
            return
        epoch = self.trainer.current_epoch  # Current epoch number
        metrics = {k: v.item() if isinstance(v, torch.Tensor) else v for k, v in self.trainer.logged_metrics.items()}  # Extract metrics
        formatted_metrics = {k: f"{v:.5f}" for k, v in metrics.items()}  # Format metrics
        print(f"Epoch {epoch}: {formatted_metrics}")  # Print metrics

In [6]:


N_folds = 5
# Load the best models
models = []
for fold in range(N_folds):
    checkpoint_path = os.path.expanduser(f'../models/{CONFIG2.model_paths[0]}/nn_{fold}.model')
    model = NN.load_from_checkpoint(checkpoint_path)
    models.append(model.to("cuda:0"))

lags_ : pl.DataFrame | None = None

def predict_nn_xgb(test: pl.DataFrame, lags: pl.DataFrame | None, batch_size: int = 1024) -> pl.DataFrame | pd.DataFrame:
    global lags_
    if lags is not None:
        lags_ = lags

    # Initialize predictions DataFrame
    predictions = test.select(
        'row_id',
        pl.lit(0.0).alias('responder_6'),
    )
    symbol_ids = test.select('symbol_id').to_numpy()[:, 0]

    # Initialize prediction array
    preds = np.zeros((test.shape[0],))

    # XGBoost prediction part
    xgb_preds = xgb_model.predict(test[xgb_feature_cols].to_numpy())
    # print(f"xgb_preds shape: {xgb_preds.shape}")
    assert xgb_preds.shape[0] == test.shape[0], "XGBoost predictions length does not match test data length"
    preds += xgb_preds / 2

    # Convert test data to PyTorch tensor
    test_input = test[CONFIG2.feature_cols].to_pandas()
    test_input = test_input.fillna(method='ffill').fillna(0)
    test_input = torch.FloatTensor(test_input.values)
    # print(f"test_input shape: {test_input.shape}")
    assert test_input.shape[0] == test.shape[0], "test_input length does not match test data length"

    # Process data in batches
    num_samples = test_input.shape[0]
    for start_idx in tqdm(range(0, num_samples, batch_size)):
        end_idx = min(start_idx + batch_size, num_samples)
        # print(f"start_idx: {start_idx}, end_idx: {end_idx}")
        assert end_idx <= num_samples, "end_idx out of range"
        batch_input = test_input[start_idx:end_idx].to("cuda:0")

        with torch.no_grad():
            for i, nn_model in enumerate(models):
                nn_model.eval()
                nn_output = nn_model(batch_input).cpu().numpy()
                # print(f"nn_output shape: {nn_output.shape}")
                assert nn_output.shape[0] == batch_input.shape[0], "Neural network output length does not match batch size"
                preds[start_idx:end_idx] += nn_output / 10

    # print(f"predict> preds.shape =", preds.shape)

    # Add predictions to DataFrame
    predictions = test.select('row_id').with_columns(
        pl.Series(
            name='responder_6',
            values=np.clip(preds, a_min=-5, a_max=5),
            dtype=pl.Float64,
        )
    )

    return predictions


# Call the function with a specified batch size
batch_size = 1024  # Adjust batch size based on GPU memory
preds = predict_nn_xgb(data, None, batch_size=batch_size)

100%|██████████| 147/147 [00:00<00:00, 208.45it/s]


In [7]:
y = np.array(data['responder_6'])

y_pred = np.array(preds['responder_6'])

# judge sign
hit = np.sum(np.sign(y) == np.sign(y_pred))

# calculate the hit rate
hit_rate = hit / len(y)
print(f"命中率: {hit_rate}")

命中率: 0.5294788056518261


In [8]:
weight = np.array(data['weight'])

r2 = r2_val(y,y_pred,weight)
print(f"r2_score: {r2}")

r2_score: 0.008418229780411624


In [9]:
preds

row_id,responder_6
u32,f64
0,0.00913
1,0.014229
2,-0.075887
3,0.222159
4,0.083984
…,…
150035,0.038305
150036,-0.088029
150037,-0.012536


In [10]:
file_path = os.path.expanduser('../preds')

# trans Polars DataFrame to Pandas DataFrame
preds_pandas = preds.to_pandas()

# save result to xgn_nn_preds.pkl
preds_pandas.to_pickle(os.path.join(file_path, 'xgn_nn_preds.pkl'))

The following is the definition of tabm model, available for further comparation.

In [11]:
def predict_tabm(test: pl.DataFrame, lags: pl.DataFrame | None, batch_size: int = 1024) -> pl.DataFrame | pd.DataFrame:
    global lags_, lags_history
    
    # Encode categorical columns
    for col in feature_cat + ['symbol_id', 'time_id']:
        test = encode_column(test, col, category_mappings[col])

    # Initialize predictions
    predictions = test.select('row_id')
    
    # Extract symbol_ids and time_ids
    symbol_ids = test.select('symbol_id').to_numpy()[:, 0]
    time_id = test.select("time_id").to_numpy()[0]
    time_id_array = test.select("time_id").to_numpy()[:, 0]
    
    # Handle lags data
    if lags is not None:
        # Ensure lags contains 'time_id' and 'symbol_id' columns
        if 'time_id' not in lags.columns or 'symbol_id' not in lags.columns:
            raise ValueError("lags DataFrame must contain 'time_id' and 'symbol_id' columns.")
        
        if time_id == 0:
            lags = lags.with_columns(pl.col('time_id').cast(pl.Int64))
            lags = lags.with_columns(pl.col('symbol_id').cast(pl.Int64))
            lags_history = lags
            lags = lags.filter(pl.col("time_id") == 0)
            test = test.join(lags, on=["time_id", "symbol_id"], how="left")
        else:
            lags = lags_history.filter(pl.col("time_id") == time_id)
            test = test.join(lags, on=["time_id", "symbol_id"], how="left")
    else:
        # If lags is None, skip the join operation
        pass

    # Fill null values
    test = test.with_columns([
        pl.col(col).fill_null(0) for col in feature_list + [f"responder_{idx}_lag_1" for idx in range(9)] 
    ])

    # Standardize features
    test = standardize(test, std_feature, means, stds)

    # Convert to numpy array
    X_test = test[feature_test].to_numpy()

    # Initialize an empty array to store predictions
    all_preds = np.zeros(len(X_test))

    # Process data in batches
    for i in range(0, len(X_test), batch_size):
        batch_X_test = X_test[i:i + batch_size]
        batch_symbol_ids = symbol_ids[i:i + batch_size]
        batch_time_id_array = time_id_array[i:i + batch_size]

        # Convert to PyTorch tensors
        X_test_tensor = torch.tensor(batch_X_test, dtype=torch.float32).to(device)
        symbol_tensor = torch.tensor(batch_symbol_ids, dtype=torch.float32).to(device)
        time_tensor = torch.tensor(batch_time_id_array, dtype=torch.float32).to(device)

        # Split into categorical and continuous features
        X_cat = X_test_tensor[:, [9, 10, 11]]
        X_cont = X_test_tensor[:, [i for i in range(X_test_tensor.shape[1]) if i not in [9, 10, 11]]]

        # Ensure all tensors have the same size in dimension 0
        if X_cat.size(0) != symbol_tensor.size(0) or X_cat.size(0) != time_tensor.size(0):
            min_size = min(X_cat.size(0), symbol_tensor.size(0), time_tensor.size(0))
            X_cat = X_cat[:min_size]
            symbol_tensor = symbol_tensor[:min_size]
            time_tensor = time_tensor[:min_size]

        # Concatenate categorical features with symbol and time IDs
        X_cat = (torch.concat([X_cat, symbol_tensor.unsqueeze(-1), time_tensor.unsqueeze(-1)], axis=1)).to(torch.int64)

        # Make predictions
        modeltrans.eval()
        with torch.no_grad():
            outputs = modeltrans(X_cont, X_cat)
            batch_preds = outputs.squeeze(-1).cpu().numpy()
            batch_preds = batch_preds.mean(1)

        # Store batch predictions
        all_preds[i:i + batch_size] = batch_preds[:len(batch_X_test)]

    # Clip predictions and create final DataFrame
    predictions = predictions.with_columns(
        pl.Series(
            name='responder_6',
            values=np.clip(all_preds, a_min=-5, a_max=5),
            dtype=pl.Float64,
        )
    )

    return predictions